In [1]:
import json
from pathlib import Path

# Find the repository from either its root or the notebooks directory.
repository_root = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "rlftqc").is_dir()
)

# Load the 17 signed stabilizer strings as the target.
target_path = repository_root / "targets" / "unmeasured-target.json"
target = json.loads(target_path.read_text(encoding="utf-8"))

print(f"Loaded {len(target)} stabilizers")
print(target)



Loaded 17 stabilizers
['+ZZXIIIIIIIIIIIIII', '+IZIXIIIIIIIIIIIII', '+XIZIIIIXIIIIIIIII', '+IXZZZIIIXIIIIIIII', '+IIIIZIIIIXIIIIIII', '+IIIIZZIIIIXIIIIII', '+IIIIIIZIIIIXIIIII', '+IIXIIIZZZIIIXIIII', '+IIIXIIIIZIIIIXIII', '+IIIIXIIIZZZIIIXII', '+IIIIIXIIIIZIIIIII', '+IIIIIIXIIIIZZIIII', '+IIIIIIIXIIIIZIIII', '+IIIIIIIIXIIIZZZXI', '+IIIIIIIIIXIIIIZIX', '+IIIIIIIIIIIIIXIZI', '+IIIIIIIIIIIIIIXZZ']


In [2]:
import numpy as np

import sys

sys.path.append(r'../')
from rlftqc.simulators.clifford_gates import CliffordGates

%load_ext autoreload
%autoreload 2
%matplotlib inline

from rlftqc.envs.logical_state_preparation_env import LogicalStatePreparationEnv
from rlftqc.envs.wrappers.tableau_to_grid_wrapper import TableauToGridWrapper
from rlftqc.envs.cells import build_cells, get_mapping, get_target
from rlftqc.logical_state_preparation import LogicalStatePreparation

print(f"len(target): {len(target)}")

cliff_gates = CliffordGates(17)

cells_from_helper = build_cells()
mapping = get_mapping()
action_mask = np.zeros((len(cells_from_helper), 5), dtype=int)
for i, cell in enumerate(cells_from_helper):
    action_mask[i, :] = np.array(cell.action_mask)


entropy = 0.05
total_timesteps = 1e5
num_envs = 10 


gates = [cliff_gates.h, cliff_gates.cx, cliff_gates.cz]
lsp = LogicalStatePreparation(target, gates=gates, cells=cells_from_helper, action_mask=action_mask, mapping=mapping)
lsp.training_config["ENT_COEF"] = entropy
lsp.training_config["TOTAL_TIMESTEPS"] = total_timesteps
lsp.training_config["NUM_ENVS"] = num_envs

len(target): 17
[[0 1]
 [0 2]
 [1 1]
 [1 2]
 [1 3]
 [1 4]
 [2 0]
 [2 1]
 [2 2]
 [2 3]
 [2 4]
 [3 0]
 [3 1]
 [3 2]
 [3 3]
 [4 2]
 [4 3]]
index_grid:
 [[-1  0  1 -1 -1]
 [-1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 -1]
 [-1 -1 15 16 -1]]


/Users/shakeebmajid/dev/magiqware/rlftqc-biased-noise/.venv312/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=uint8 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [ ]:
# Log each completed episode’s total reward and number of steps.
# Run wandb login in your training environment before using online mode.
# Use mode="offline" to save locally, or wandb_options=None to disable tracking.
lsp.train(wandb_options={
    "project": "rlftqc-biased-noise",
    "name": "target-training",
    "mode": "online",
    "config": {
        "entropy": entropy,
        "total_timesteps": total_timesteps,
        "num_envs": num_envs
    }
})

==== Training begin


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/shakeebmajid/.netrc.
wandb: Currently logged in as: shakeeb-majid (shakeeb-majid-magiqware) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
